# Cleaning Data

## Did you like the movie? - Cleaning the database

### Loading libraries

The libraries we will use for cleaning our dataset are `pandas` (for working with the data set itself), `numpy` (for working with NAs and other array computations) and `re` (so we can use regular expressions for cleaning certain columns).

##### First of all, we import the libraries nedded to manipulate the database

In [ ]:
import pandas as pd
import numpy as np
from numpy import nan
np.set_printoptions(suppress=True)

### Loading the dataset

The first order of business is loading the dataset into a `pandas DataFrame` object.

##### Here we define the intended column names and loads the dataset into a pandas DataFrame

In [ ]:
column_names = ["Title", "Year", "Runtime", "Genre", "Budget", "Revenue", "Personal", "IMDb", "Rotten_Tomatoes", "Metacritic",
                "Filmaffinity", "P1", "P2", "P3", "P4", "P5", "P6", "P7", "P8", "P9", "P10", "P11", "P12", "P13",
                "P14", "P15", "P16", "P17", "P18", "P19", "P20"]

team1 = pd.read_csv('Team1_movies_dataset-Copy1.csv')

Let’s explore our database so we can get a glimpse at the data

##### We show the first rows of the DataFrame

In [ ]:
team1.head()

,Title,Year,Runtime,Genre,Budget,Revenue,Personal,IMDb,Rotten_Tomatoes,Metacritic,...,P11,P12,P13,P14,P15,P16,P17,P18,P19,P20
0,The Godfather (1972),1972,175,Gangster,6.0,250.0,96,92,97,90,...,90.0,NaN,NaN,NaN,NaN,NaN,NaN,90.0,95.0,NaN
1,The Godfather Part II (1974),1974,202,Gangster,13.0,93.0,94,90,96,88,...,95.0,NaN,NaN,NaN,NaN,NaN,NaN,88.0,80.0,100.0
2,The Dark Knight (2008),2008,152,Superhero,185.0,1000.0,100,91,94,89,...,100.0,NaN,100.0,NaN,80.0,NaN,NaN,99.0,100.0,70.0
3,Pulp Fiction (1994),1994,154,Gangster,8.0,213.0,84,88,92,86,...,90.0,NaN,NaN,NaN,NaN,NaN,NaN,80.0,NaN,85.0
4,The Shawshank Redemption (1994),1994,142,Drama,25.0,28.0,84,93,89,91,...,100.0,NaN,NaN,NaN,NaN,NaN,NaN,99.0,NaN,80.0


We can check the dimensions of our dataframe with `pd.shape`.

##### We prints the dimensions of the dataset

In [ ]:
team1.shape

(200, 31)

We observe that there are 200 observations with 31 attributes.

### Column cleaning

In this section we will clean the column names so they look homogeneous, without extra (invisible) characters.

First of all, we print a list of column names.

##### We lists all column names to check for inconsistencies, typos, or formatting problems

In [ ]:
team1.columns

Index(['Title', 'Year', 'Runtime', 'Genre', 'Budget', 'Revenue', 'Personal',
       'IMDb', 'Rotten_Tomatoes', 'Metacritic', 'Filmaffinity', 'P1', 'P2',
       'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10', 'P11', 'P12', 'P13',
       'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20'],
      dtype='object')

All column names are fine, but we do not need the first columns, so we drop it.

##### We remove unnecessary columns, selects the desired ones, and renames all columns to ensure consistency across the dataset

In [ ]:
team1 = team1[['Movie', 'Year', 'Runtime( minutes)', 'Genre', 'Budget (Millions USD)',
       'Revenue (Millions USD)','Personal ', 'IMDb', 'Rotten Tomatoes', 'Metacritic',
       'Filmaffinnity', 'Person 1', 'Person 2', 'Person 3',
       'Person 4', 'Person 5', 'Person 6', 'Person 7', 'Person 8', 'Person 9',
       'Person 10', 'Person 11', 'Person 12', 'Person 13', 'Person 14',
       'Person 15', 'Person 16', 'Person 17', 'Person 18', 'Person 19',
       'Person 20']]
team1.columns = column_names

team1.columns

KeyError: "['Movie', 'Runtime( minutes)', 'Budget (Millions USD)', 'Revenue (Millions USD)', 'Personal ', 'Rotten Tomatoes', 'Filmaffinnity', 'Person 1', 'Person 2', 'Person 3', 'Person 4', 'Person 5', 'Person 6', 'Person 7', 'Person 8', 'Person 9', 'Person 10', 'Person 11', 'Person 12', 'Person 13', 'Person 14', 'Person 15', 'Person 16', 'Person 17', 'Person 18', 'Person 19', 'Person 20'] not in index"

Finally, we will set the `Title` column as the indexing column of our dataframe.

##### We set the 'Title' column as the index of the DataFrame for easier row identification

In [ ]:
team1.set_index('Title', inplace=True)

In [ ]:
team1.info()

In [ ]:
team1.columns

### Data cleaning

We will follow the guidelines stated below for cleaning the data for each column:
- `Year`: This column will be left unchanged for the time being. Data type can be `str` or `int`. It it isn't, we will recast it later.
- `Runtime`: Data type should be `int`. Remove any indication to a unit of measurement.
- `Genre`: Data type should be `str`. We will capitalize the data, and if there is more than one genre, we will keep the first one and remove the rest.
- `Budget (Millions USD)`: Data type can be `int` or `float`. To that end, we will remove dollar signs as well as the thousands separator. We will scale the data so the unit of measurment is millions of dollars.
- `Revenue (Millions USD)`: Same treatment as that of `Budget (Millions USD)`.
- Rating columns: Data type can be `int` or `float`. Change scale to [0,100].

Let's explore the type of each of the columns. In the table below we can also see if there is missing data.

In [ ]:
team1.info()

##### We lists the columns stored as objects to detect which numeric fields were incorrectly imported as text


In [ ]:
team1.select_dtypes(include='object').columns

We observe that columns `Budget (Millions USD)`, `Revenue (Millions USD)`, `Personal`, `P15`, `P16`, `P17`, and `P18` have the wrong type. Let's explore each column in turn, and convert them to `int` or `float` as appropriate, except `Year`. Ratings smaller than 10 will be multiplied by 10.

##### We identifie all rating-related columns


In [ ]:
rating_columns=team1.columns[5:]
rating_columns

Index(['Revenue', 'Personal', 'IMDb', 'Rotten_Tomatoes', 'Metacritic',
       'Filmaffinity', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9',
       'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19',
       'P20'],
      dtype='object')

#### `Runtime`

##### Here we inspect the values in the 'Runtime' column to detect incorrect unit

In [ ]:
np.sort(team1['Runtime'].unique())

array([ 91,  92,  93,  95,  96,  98, 100, 101, 102, 103, 104, 106, 107,
       108, 109, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121,
       122, 123, 124, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135,
       136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 147, 148, 149,
       150, 151, 152, 153, 154, 155, 156, 158, 161, 162, 164, 166, 169,
       175, 176, 180, 181, 187, 192, 194, 195, 202, 242])

#### `Budget (Millions USD)` and `Revenue (Millions USD)`

##### We explore the values in the 'Budget' column to identify incorrect formats

In [ ]:
team1['Budget'].unique()

array([  6.  ,  13.  , 185.  ,   8.  ,  25.  ,  68.  , 160.  ,  63.  ,
       150.  , 165.  ,  40.  ,  55.  , 350.  , 100.  ,  11.  , 200.  ,
        60.  ,  50.  , 177.  , 237.  , 400.  ,  18.5 , 130.  ,  97.  ,
       110.  ,  65.  , 300.  , 375.  , 415.  , 149.  , 125.  , 175.  ,
       139.  , 383.  , 230.  ,  90.  , 135.  , 222.5 , 365.  , 362.5 ,
       378.  , 178.35, 173.5 , 250.  , 170.  , 180.  , 200.8 , 149.65,
       178.5 , 163.5 ,  58.  ,    nan,  75.  , 210.  , 202.5 , 178.  ,
        45.  ,  54.  ,  67.5 ,  22.  ,  74.  ,  93.  , 182.5 , 325.  ,
        19.  ,  10.5 ,  20.  ,  18.  ,  23.  ,  28.2 ,  48.  , 341.  ,
        25.5 ,  37.6 , 115.  , 113.  , 376.5 , 317.  , 275.  , 240.1 ,
       287.5 ,  28.  , 183.  , 207.5 ,  94.  ,  30.  , 120.  ,  92.  ,
         3.3 ,   4.5 ,   3.  ,   2.5 ,  15.  ,  19.6 ,  19.5 ,  66.  ,
        85.  , 190.  , 207.  ,  28.5 ,  47.  ,  46.  ])

##### We basically just need to remove dollar signs and cast as `float`.

In [ ]:
team1[team1["Budget"] == "200-317"].index.values
team1.loc["Star Wars: The Last Jedi (2017)", "Budget"] = 317

In [ ]:
team1["Budget"] = pd.to_numeric(team1["Budget"], errors = 'coerce')
np.sort(team1["Budget"].unique())

array([  2.5 ,   3.  ,   3.3 ,   4.5 ,   6.  ,   8.  ,  10.5 ,  11.  ,
        13.  ,  15.  ,  18.  ,  18.5 ,  19.  ,  19.5 ,  19.6 ,  20.  ,
        22.  ,  23.  ,  25.  ,  25.5 ,  28.  ,  28.2 ,  28.5 ,  30.  ,
        37.6 ,  40.  ,  45.  ,  46.  ,  47.  ,  48.  ,  50.  ,  54.  ,
        55.  ,  58.  ,  60.  ,  63.  ,  65.  ,  66.  ,  67.5 ,  68.  ,
        74.  ,  75.  ,  85.  ,  90.  ,  92.  ,  93.  ,  94.  ,  97.  ,
       100.  , 110.  , 113.  , 115.  , 120.  , 125.  , 130.  , 135.  ,
       139.  , 149.  , 149.65, 150.  , 160.  , 163.5 , 165.  , 170.  ,
       173.5 , 175.  , 177.  , 178.  , 178.35, 178.5 , 180.  , 182.5 ,
       183.  , 185.  , 190.  , 200.  , 200.8 , 202.5 , 207.  , 207.5 ,
       210.  , 222.5 , 230.  , 237.  , 240.1 , 250.  , 275.  , 287.5 ,
       300.  , 317.  , 325.  , 341.  , 350.  , 362.5 , 365.  , 375.  ,
       376.5 , 378.  , 383.  , 400.  , 415.  ,    nan])

##### We displays the values in the 'Revenue' column to detect issues

In [ ]:
team1['Revenue'].unique()

array([ 250. ,   93. , 1000. ,  213. ,   28. ,  101. ,  839. ,  467. ,
        741.8,  427.3,  773. ,  109. ,  678. ,  600. ,  975. ,  955. ,
        262. ,  490. ,  484. ,  549.4, 1490. ,  216.8, 2260. , 2900. ,
       2300. ,  131. ,  100. ,  159.8,  403. ,  240.9,  465.5,  456. ,
        650. ,  668. ,  873. ,  821. ,  169. , 1100. ,  271.4,  367. ,
        134. ,  749. ,  168. ,  772.3,  784.6,  895. ,  758.5,  709. ,
        880.9, 1133. ,  375.5, 1921. ,  690.8,  585.8,  623.9, 1215. ,
       1521. , 1405. , 2052. , 2799. ,  370.6,  714.4, 1153. ,  449.3,
        644.8,  855. ,  760.9,  677.8,  955.7,  519.3,  622.7,  476. ,
        770.9,  863. ,  845.6, 1349.9,  859.2, 1131.4,  402.1,  432.2,
        782.8,  785. ,    nan,  619. ,  296.3,  407. ,  459.3,  373. ,
        352.6,  746. ,  543.9,  252.4,  131.2,  155. ,  132. ,  219. ,
        482. ,  321.2, 1058. ,  618.6,  368.8, 1671. , 1310. , 1004. ,
        388.8,  336.5,  245.1,  792.9,  476.5,  208.9,   88. ,   51.9,
      

As with `Budget (Millions USD)`, we need to remove dollar signs and commas, replace missing data with `nan`, and cast as `float`. And there is a cell that contains a range, rather than a data point. Let's find what record contains it.

##### Here we replace an incorrect revenue range with an accurate numerical value

In [ ]:
team1[team1['Revenue']=='538.4–549.0'].index.values

array([], dtype=object)

According to **IMDb**, the Revenue (Millions USD) for this movie is \$550,016,064, so we will replace the given range with 550.016

In [ ]:
team1.loc['Star Wars: The Empire Strikes Back (1980)','Revenue'] = 550

##### Here we remove all commas from the values in the "Revenue" column.

In [ ]:
team1["Revenue"] = team1["Revenue"].str.replace(",", "")

AttributeError: Can only use .str accessor with string values!

##### Converts the values in the "Revenue" column into numeric data

In [ ]:
team1["Revenue"]=pd.to_numeric(team1["Revenue"], errors = 'coerce')
np.sort(team1["Revenue"].unique())

Now, we clean the rest of the column

In [ ]:
team1.describe()

#### Ratings

##### This loop goes through every column listed in rating_columns and prints all the unique values found in each one

In [ ]:
for col in rating_columns:
    print(f"{col} = {team1[col].unique()}")

##### Here we identifie which rating columns contain non-numeric values

In [ ]:
nonnumerical_rating_columns=team1[rating_columns].select_dtypes(include='object').columns
nonnumerical_rating_columns

##### Here we go through each column listed in nonnumerical_rating_columns and prints all the unique values found in those columns

In [ ]:
for col in nonnumerical_rating_columns:
    print(f"{col} = {team1[col].unique()}")

##### This line searches the dataset for rows where the P18 rating column contains the incorrect value "5580" and returns the index of those rows

In [ ]:
team1[team1["P18"]=="5580"].index.values

##### This line fixes an incorrect rating value for the movie "Alien 3 (1992)" in the P18 column

In [ ]:
team1.loc["Alien 3 (1992)", "P18"] = 80

##### Converts all remaining rating columns from text to numeric values

In [ ]:
for col in nonnumerical_rating_columns:
    team1[col]=pd.to_numeric(team1[col],errors='coerce')
for col in rating_columns:
    print(f"{col} = {np.sort(team1[col].unique())}")

Revenue = [   3.6    5.4   20.1   21.5   28.    41.8   44.2   50.4   51.    51.9
   57.6   60.5   69.9   76.    83.1   84.9   88.    93.    99.3  100.
  101.   103.2  109.   121.9  122.1  131.   131.2  132.   134.   135.3
  155.   159.8  160.3  168.   169.   195.9  203.4  208.9  213.   216.8
  219.   240.9  245.1  250.   252.4  258.   261.3  262.   271.4  276.
  296.3  304.3  321.2  330.   332.2  333.1  336.5  352.6  363.3  367.
  368.8  370.6  373.   375.5  383.   384.   385.9  388.8  389.9  393.2
  394.4  402.1  403.   404.2  407.   411.   427.3  432.2  449.3  451.2
  456.   459.3  461.   465.5  467.   467.9  474.2  476.   476.5  482.
  484.   490.   504.1  505.   511.   519.3  521.   529.   532.9  540.
  543.9  549.4  550.   559.   566.7  577.   585.8  600.   618.6  619.
  622.7  623.   623.9  631.7  644.8  650.   653.8  668.   677.8  678.
  687.2  690.8  709.   714.4  722.6  735.   741.8  743.   746.   749.
  758.5  760.9  770.9  772.3  773.   775.8  782.8  784.6  785.   790.7
  79

We suspect that ratings smaller than $5$ are in a scale from $0$ to $5$, and ratings between $5$ and $10$ are in a scale from $0$ to $10$ so we multiply them by the proper factor to bring them up to a scale from $0$ to $100$.

In [ ]:
for col in rating_columns:
    print(team1[(team1[col]>0)&(team1[col]<=10)][col].sort_values())

171    3.6
178    5.4
Name: Revenue, dtype: float64
Series([], Name: Personal, dtype: float64)
Series([], Name: IMDb, dtype: float64)
113    2.0
Name: Rotten_Tomatoes, dtype: float64
Series([], Name: Metacritic, dtype: float64)
Series([], Name: Filmaffinity, dtype: float64)
Series([], Name: P1, dtype: float64)
24    10.0
Name: P2, dtype: float64
40    10.0
Name: P3, dtype: float64
Series([], Name: P4, dtype: float64)
40    10.0
Name: P5, dtype: float64
Series([], Name: P6, dtype: float64)
37     1.0
34     4.0
22    10.0
38    10.0
Name: P7, dtype: float64
34    10.0
Name: P8, dtype: float64
15    5.0
38    5.0
Name: P9, dtype: float64
Series([], Name: P10, dtype: float64)
Series([], Name: P11, dtype: float64)
Series([], Name: P12, dtype: float64)
Series([], Name: P13, dtype: float64)
Series([], Name: P14, dtype: float64)
Series([], Name: P15, dtype: float64)
Series([], Name: P16, dtype: float64)
Series([], Name: P17, dtype: float64)
Series([], Name: P18, dtype: float64)
38    1.0
15  

##### Locates the row labeled "Superman (2025)" and updates the value in its "Filmaffinity" column to 63

In [ ]:
team1.loc["Superman (2025)", "Filmaffinity"] = 63

In [ ]:
team1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 204 entries, 0 to Superman (2025)
Data columns (total 31 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Title            200 non-null    object 
 1   Year             200 non-null    float64
 2   Runtime          200 non-null    float64
 3   Genre            200 non-null    object 
 4   Budget           198 non-null    float64
 5   Revenue          197 non-null    float64
 6   Personal         200 non-null    float64
 7   IMDb             200 non-null    float64
 8   Rotten_Tomatoes  200 non-null    float64
 9   Metacritic       200 non-null    float64
 10  Filmaffinity     201 non-null    float64
 11  P1               99 non-null     float64
 12  P2               125 non-null    float64
 13  P3               168 non-null    float64
 14  P4               168 non-null    float64
 15  P5               150 non-null    float64
 16  P6               169 non-null    float64
 17  P7       

#### `Genre`

Now, let's explore the `'Genre'` column to see if there are invisible characters.

In [ ]:
np.sort(team1['Genre'].unique())

TypeError: '<' not supported between instances of 'float' and 'str'

There are trailing white spaces on some entries, as well as a few spelling mistakes, duplicate genres, and a `nan`. Let's clean this column.

In [ ]:
team1['Genre'] = team1['Genre'].str.replace(' ','')
np.sort(team1['Genre'].unique())

##### This cell removes all spaces from the 'Genre' column and then prints the sorted list of unique genre values to verify that the cleaning worked correctly

In [ ]:
team1['Genre'] = team1['Genre'].replace('superhero','Superhero')
team1['Genre'] = team1['Genre'].replace('Aventure','Adventure')
np.sort(team1['Genre'].unique())

Our data is clean, with the correct data types.

In [ ]:
team1.info()

Now, let's verify that the rating columns have the correct scale

In [ ]:
team1.iloc[:,3:10].describe()

In [ ]:
team1.iloc[:,10:20].describe()

In [ ]:
team1.iloc[:,20:].describe()

### Save dataframe to disk

In [ ]:
team1.to_csv('team1_clean.csv')